# 5 · Ejercicio 4: abstraer un patrón (20 min)

El enunciado te da 2-3 funciones "basadas en un mismo patrón de diseño" y pide:
- **a)** una función de orden superior que encapsule el patrón,
- **b)** reescribir las funciones usándola (una línea cada una).

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen, comentadas.

In [ ]:
// ─────────────────────────────────────────────────────────────────────────────
// PREÁMBULO — las mismas definiciones que te dan en el examen.
// Ejecuta esta celda antes que ninguna otra del notebook.
// ─────────────────────────────────────────────────────────────────────────────

// Librería de tests. `run(TestX(f))` ejecuta los tests de un apartado.
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.
type Not[P] = P => Nothing
// "P si y solo si Q" = un par de funciones: de P a Q y de Q a P.
type <=>[P, Q] = (P => Q, Q => P)

// Tercio excluso: para cualquier P, o tengo una P o tengo una prueba de que no hay P.
// Se usa escribiendo  lem[X]  y haciendo match sobre el Either que devuelve.
trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

// Doble negación: si "no es verdad que no P", entonces P.
// Se usa escribiendo  dn[X](nx => ...)  cuando la meta es X.
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

// Plantilla de los isomorfismos: tú implementas from y to; las leyes las usan los tests.
trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)   // ir y volver deja igual una A
    def law2(b: B): Boolean = equalB(from(to(b)), b)   // ir y volver deja igual una B

// Árbol binario: o vacío, o un nodo con hijo izquierdo, raíz e hijo derecho.
enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

// Atajos para escribir árboles en los tests. El orden de los argumentos imita el dibujo:
// left(hijo, raiz) pone el hijo a la izquierda; right(raiz, hijo) a la derecha.
object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)

import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)

## La receta

1. Pon las funciones una al lado de otra. Marca lo **igual** (la estructura) y lo que **cambia**.
2. Lo que cambia → **parámetro función** (`f: A => B`, `f: (A, B) => C`…).
3. Los tipos concretos (`Int`, `String`…) → genéricos (`A`, `B`, `C`).
4. Copia la estructura llamando a `f` donde había código concreto. En la llamada recursiva pasa también `f`.
5. Cada función original = la HOF + una lambda.

Pon la estructura en una lista de parámetros y las funciones en otra: `def hof[A, B](datos: ...)(f: ...)`.

## Ejemplo (examen 25-26) · `mapLeft`

In [ ]:
// Las dos funciones que da el enunciado. Compáralas línea a línea:
def add1(t: Either[Int, String]): Either[Int, String] =
    t match
        case Left(i)  => Left(i + 1)     // ← lo único distinto: "i + 1"
        case Right(s) => Right(s)        // ← igual: el Right se devuelve tal cual

def not(t: Either[Boolean, Char]): Either[Boolean, Char] =
    t match
        case Left(b)  => Left(!b)        // ← lo único distinto: "!b"
        case Right(c) => Right(c)        // ← igual

Igual: el `match`; el `Right` se devuelve tal cual. Cambia: lo que se le hace al valor del `Left` (`i + 1`, `!b`) → `f: A => C`.

In [ ]:
// Paso 2: lo que cambia ("lo que se le hace al Left") pasa a ser un parámetro f.
// Paso 3: los tipos concretos pasan a ser genéricos:
//   A = tipo del Left (Int en add1, Boolean en not)
//   B = tipo del Right (String, Char): no se toca, así que sale igual
//   C = tipo de lo que devuelve f (podría ser distinto de A, así que le doy otro nombre)
// La estructura (t) va en una lista de parámetros y la función (f) en otra:
// así Scala deduce A y B mirando t, y luego ya sabe los tipos de la lambda f.
def mapLeft[A, B, C](t: Either[A, B])(f: A => C): Either[C, B] =
    // Paso 4: copio la estructura común, poniendo f(a) donde estaba "i + 1" / "!b".
    t match
        case Left(a)  => Left(f(a))
        case Right(b) => Right(b)

// Paso 5: cada función original es ahora una línea: mapLeft + "lo que cambiaba".
def add1(t: Either[Int, String]): Either[Int, String] = mapLeft(t)(_ + 1)   // f = x => x + 1
def not(t: Either[Boolean, Char]): Either[Boolean, Char] = mapLeft(t)(!_)  // f = x => !x

(add1(Left(1)), not(Left(true)), add1(Right("x")))

## Las cuatro familias que salen

| Familia | Idea | Signatura típica |
|---|---|---|
| **map** | misma forma, se transforma el contenido | `def map[A, B](t: F[A])(f: A => B): F[B]` |
| **fold** | se recorre y se calcula un resultado | `def fold[A, B](t: F[A])(base: B)(combinar: ...): B` |
| **unfold** | se **genera** desde una semilla | `def unfold[A, B](seed: B)(next: B => Option[(A, B)]): List[A]` |
| **traverse** | como map, pero `f` puede fallar (`Option`); si falla una, falla todo | `def traverse[A, B](l: List[A])(f: A => Option[B]): Option[List[B]]` |

### Ejemplo de familia fold: árboles

In [ ]:
// Funciones como size o sum sobre árboles tienen todas esta forma:
//   case Empty()       => (un valor fijo: 0)
//   case Node(l, x, r) => (combinar el resultado de l, la raíz x y el resultado de r)
// Lo que cambia: el valor para Empty (empty) y cómo combinar (node).
def foldT[A, B](t: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    t match
        case Empty() => empty
        case Node(l, x, r) =>
            // Primero resuelvo los dos subárboles (con los MISMOS parámetros empty y node:
            // si no se los paso, la recursión no sabría qué hacer) y luego combino.
            node(foldT(l)(empty)(node), x, foldT(r)(empty)(node))

// size: un árbol vacío tiene 0 nodos; un nodo tiene los de la izquierda + 1 + los de la derecha.
def size[A](t: Tree[A]): Int = foldT(t)(0)((l, _, r) => l + 1 + r)
// sum: igual, pero sumando el valor de la raíz en vez de 1.
def sumT(t: Tree[Int]): Int  = foldT(t)(0)((l, x, r) => l + x + r)

val t = node(leaf(1), 2, right(3, leaf(4)))    //   2
(size(t), sumT(t))                              //  / \
                                                // 1   3
                                                //      \
                                                //       4      → (4, 10)

### Ejemplo de familia traverse

In [ ]:
// Aplica f a cada elemento. f puede fallar (devolver None). Si TODOS salen bien,
// devuelve Some(lista de resultados); si alguno falla, None.
def traverse[A, B](l: List[A])(f: A => Option[B]): Option[List[B]] =
    l match
        case Nil => Some(Nil)       // una lista vacía no tiene nada que falle → éxito con lista vacía
        case head :: tail =>
            // Calculo a la vez el resultado de la cabeza y el del resto (recursión, pasando f)
            // y miro las dos cosas juntas con un match sobre el par:
            (f(head), traverse(tail)(f)) match
                case (Some(b), Some(bs)) => Some(b :: bs)   // los dos bien: junto los resultados
                case _ => None                              // cualquier otro caso: algo falló

// "12".toIntOption da Some(12); "x".toIntOption da None.
def parse(l: List[String]): Option[List[Int]] = traverse(l)(_.toIntOption)

(parse(List("1", "2")), parse(List("1", "x")))     // (Some(List(1, 2)), None)

## Mini-ejercicio (5 min)

Abstrae el patrón en `mapOption` y reimplementa ambas funciones.

In [ ]:
def inc(o: Option[Int]): Option[Int] =
    o match
        case None => None
        case Some(n) => Some(n + 1)

def len(o: Option[String]): Option[Int] =
    o match
        case None => None
        case Some(s) => Some(s.length)

In [ ]:
def mapOption[A, B](o: Option[A])(f: A => B): Option[B] =
    ???

<details>
<summary><b>Ver solución</b></summary>

```scala
// Igual en las dos: None → None; Some(x) → Some(algo con x).
// Cambia: "algo con x" (n + 1, s.length) → f.  Tipos: A = lo de dentro, B = lo que devuelve f.
def mapOption[A, B](o: Option[A])(f: A => B): Option[B] =
    o match
        case None => None
        case Some(a) => Some(f(a))

def inc(o: Option[Int]): Option[Int] = mapOption(o)(_ + 1)
def len(o: Option[String]): Option[Int] = mapOption(o)(_.length)
```

</details>

## ¡Listo!

Ahora haz un simulacro con reloj: `examenes-prueba/Prueba1-Enunciado.ipynb` (1h30m) y corrige con `Prueba1-Solucion.ipynb`. Si te atascas en algo, el detalle está en `claudetemario`.